<a href="https://colab.research.google.com/github/henryzhou24/IDX-Exchange-Data-Analyst/blob/main/IDX_Weeks4_5_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 4–5: Data Cleaning and Preparation

## Objective

This notebook prepares the IDX Exchange MLS datasets for reliable analysis by standardizing data types, reviewing missing values, identifying invalid numeric values, validating date consistency, and checking geographic coordinates.

## Input Data

- Residential Sold dataset with mortgage-rate enrichment
- Residential Listings dataset with mortgage-rate enrichment

## Data Integrity Principles

1. Preserve the original input files.
2. Document every transformation and the reason for performing it.
3. Record row counts before and after cleaning.
4. Flag questionable records before removing them.
5. Validate the final datasets and retain the `rate_30yr_fixed` field.

## Workflow

1. Load and inspect the datasets.
2. Audit columns, missing values, and data types.
3. Standardize date and numeric fields.
4. Review redundant columns and missing values.
5. Flag invalid numeric values.
6. Check date consistency.
7. Validate geographic coordinates.
8. Export cleaned datasets and quality reports.

In [1]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount("/content/drive")

# Define project paths
BASE_DIR = "/content/drive/MyDrive/IDX-Exchange"
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

SOLD_FILE = os.path.join(
    OUTPUT_DIR,
    "CRMLSSold_202401_202604_Residential_WithMortgageRate.csv"
)

LISTINGS_FILE = os.path.join(
    OUTPUT_DIR,
    "CRMLSListing_202401_202604_Residential_WithMortgageRate.csv"
)

# Confirm that both input files exist
for file_path in [SOLD_FILE, LISTINGS_FILE]:
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 ** 2)
        print(f"FOUND: {os.path.basename(file_path)} ({size_mb:.1f} MB)")
    else:
        print(f"NOT FOUND: {file_path}")

Mounted at /content/drive
FOUND: CRMLSSold_202401_202604_Residential_WithMortgageRate.csv (231.4 MB)
FOUND: CRMLSListing_202401_202604_Residential_WithMortgageRate.csv (295.1 MB)


## Step 1: Initial Dataset Inspection

Before cleaning, we inspect the datasets to establish a baseline. This step records the original row counts, column names, data types, and missing-value counts.

We will use this baseline to determine which fields require conversion or further investigation and to verify the impact of subsequent cleaning operations.

No records or columns are removed in this step.

In [2]:
# Load the datasets without modifying the source files
sold = pd.read_csv(SOLD_FILE, low_memory=False)
listings = pd.read_csv(LISTINGS_FILE, low_memory=False)

datasets = {
    "Sold": sold,
    "Listings": listings
}

# Record baseline row and column counts
print("=== BASELINE DATASET SUMMARY ===")

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")
    print(f"Duplicate column names: {df.columns.duplicated().sum():,}")
    print(f"Exact duplicate rows: {df.duplicated().sum():,}")

# Check whether the mortgage-rate column exists
print("\n=== MORTGAGE RATE CHECK ===")

for name, df in datasets.items():
    print(
        f"{name}: rate_30yr_fixed present = "
        f"{'rate_30yr_fixed' in df.columns}"
    )

=== BASELINE DATASET SUMMARY ===

Sold
Rows: 414,003
Columns: 86
Duplicate column names: 0
Exact duplicate rows: 0

Listings
Rows: 547,162
Columns: 86
Duplicate column names: 0
Exact duplicate rows: 0

=== MORTGAGE RATE CHECK ===
Sold: rate_30yr_fixed present = True
Listings: rate_30yr_fixed present = True


In [3]:
# Fields specifically required by the handbook
date_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

numeric_fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket",
    "Bedrooms",
    "Bathrooms",
    "Latitude",
    "Longitude",
    "latfilled",
    "lonfilled",
    "rate_30yr_fixed"
]

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} — COLUMN AUDIT")
    print("=" * 60)

    print("\nDate fields:")
    for col in date_fields:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")
        else:
            print(f"  {col}: NOT PRESENT")

    print("\nNumeric and geographic fields:")
    for col in numeric_fields:
        if col in df.columns:
            print(f"  {col}: {df[col].dtype}")
        else:
            print(f"  {col}: NOT PRESENT")

    print("\nFirst 30 column names:")
    print(df.columns.tolist()[:30])


SOLD — COLUMN AUDIT

Date fields:
  CloseDate: object
  PurchaseContractDate: object
  ListingContractDate: object
  ContractStatusChangeDate: object

Numeric and geographic fields:
  ClosePrice: float64
  LivingArea: float64
  DaysOnMarket: int64
  Bedrooms: NOT PRESENT
  Bathrooms: NOT PRESENT
  Latitude: float64
  Longitude: float64
  latfilled: object
  lonfilled: object
  rate_30yr_fixed: float64

First 30 column names:
['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency']

LISTINGS — COLUMN AUDIT

Date fields:


## Step 2: Inspect Relevant MLS Fields

MLS datasets may use field names that differ from generic descriptions in the handbook. For example, bedroom and bathroom counts may be stored in fields such as `BedroomsTotal` or `BathroomsTotal`.

This section identifies the actual fields in each dataset and inspects representative values. We will use these findings to define the cleaning rules consistently across Sold and Listings records.

No data is modified in this step.

In [4]:
# Find columns related to dates, bedrooms, bathrooms, coordinates,
# and other fields relevant to cleaning.

search_terms = [
    "date",
    "bedroom",
    "bathroom",
    "latitude",
    "longitude",
    "latfilled",
    "lonfilled",
    "livingarea",
    "daysonmarket",
    "closeprice",
    "purchasecontract",
    "listingcontract",
    "contractstatuschange"
]

for name, df in datasets.items():
    print(f"\n{'=' * 65}")
    print(f"{name.upper()} — RELEVANT COLUMN NAMES")
    print("=" * 65)

    for term in search_terms:
        matches = [
            col for col in df.columns
            if term in col.lower()
        ]
        if matches:
            print(f"{term}: {matches}")

    # Inspect selected geographic columns
    geo_cols = [
        col for col in [
            "Latitude", "Longitude", "latfilled", "lonfilled"
        ]
        if col in df.columns
    ]

    print("\nGeographic field examples:")
    if geo_cols:
        print(df[geo_cols].head(10).to_string(index=False))

    # Inspect actual date values before conversion
    actual_date_cols = [
        col for col in [
            "CloseDate",
            "PurchaseContractDate",
            "ListingContractDate",
            "ContractStatusChangeDate"
        ]
        if col in df.columns
    ]

    print("\nDate field examples:")
    print(df[actual_date_cols].head(5).to_string(index=False))


SOLD — RELEVANT COLUMN NAMES
date: ['CloseDate', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate']
bedroom: ['BedroomsTotal', 'MainLevelBedrooms']
bathroom: ['BathroomsTotalInteger']
latitude: ['Latitude']
longitude: ['Longitude']
latfilled: ['latfilled']
lonfilled: ['lonfilled']
livingarea: ['LivingArea']
daysonmarket: ['DaysOnMarket']
closeprice: ['ClosePrice']
purchasecontract: ['PurchaseContractDate']
listingcontract: ['ListingContractDate']
contractstatuschange: ['ContractStatusChangeDate']

Geographic field examples:
 Latitude   Longitude latfilled lonfilled
37.566106 -122.327954      True      True
32.659315 -117.096922      True      True
32.659284 -117.097174      True      True
35.277199 -120.646786     False     False
39.419080 -123.814842     False     False
37.325454 -121.780303      True      True
32.715089 -117.170568     False     False
34.070215 -118.258618     False     False
34.107850 -118.219273     False     False
37.589762 -122.399887    

## Step 3: Audit Potentially Redundant Columns

During the initial inspection, several Listings columns were found with `.1` suffixes. These columns may be duplicated fields created during an earlier data-processing step, but we must verify that before removing them.

For each matching pair, we compare missing-value patterns and values row by row. We also inspect the percentage of missing values in key cleaning fields.

This is an audit only. No columns or records are removed.

In [5]:
# Compare columns such as Latitude and Latitude.1.
# A suffix alone is not enough to justify removing a column.

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — POTENTIAL DUPLICATE COLUMN AUDIT")
    print("=" * 70)

    suffix_cols = [col for col in df.columns if col.endswith(".1")]

    if not suffix_cols:
        print("No columns ending in '.1' found.")
        continue

    for duplicate_col in suffix_cols:
        base_col = duplicate_col[:-2]

        if base_col not in df.columns:
            print(f"\n{duplicate_col}: no matching base column")
            continue

        left = df[base_col]
        right = df[duplicate_col]

        both_present = left.notna() & right.notna()
        either_present = left.notna() | right.notna()

        matching = (
            left.eq(right) | (left.isna() & right.isna())
        )

        comparable_count = int(both_present.sum())
        mismatch_count = int(
            (both_present & ~left.eq(right)).sum()
        )

        print(f"\nPair: {base_col} vs {duplicate_col}")
        print(f"  Base column missing: {left.isna().sum():,}")
        print(f"  Suffix column missing: {right.isna().sum():,}")
        print(f"  Both values present: {comparable_count:,}")
        print(f"  Both present but different: {mismatch_count:,}")
        print(f"  Equal including matching missing values: "
              f"{int(matching.sum()):,} / {len(df):,}")

        if mismatch_count > 0:
            print("  Examples of differences:")
            examples = df.loc[
                both_present & ~left.eq(right),
                [base_col, duplicate_col]
            ].head(5)
            print(examples.to_string(index=False))

# Inspect missingness in fields relevant to the cleaning task.
key_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude",
    "rate_30yr_fixed"
]

for name, df in datasets.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — KEY FIELD COMPLETENESS")
    print("=" * 70)

    rows = []
    for col in key_fields:
        if col in df.columns:
            missing_count = int(df[col].isna().sum())
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(df) * 100, 2
                )
            })

    completeness = pd.DataFrame(rows)
    print(completeness.to_string(index=False))


SOLD — POTENTIAL DUPLICATE COLUMN AUDIT
No columns ending in '.1' found.

LISTINGS — POTENTIAL DUPLICATE COLUMN AUDIT

Pair: PropertyType vs PropertyType.1
  Base column missing: 0
  Suffix column missing: 0
  Both values present: 547,162
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: ListAgentFirstName vs ListAgentFirstName.1
  Base column missing: 4,354
  Suffix column missing: 4,354
  Both values present: 542,808
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: DaysOnMarket vs DaysOnMarket.1
  Base column missing: 0
  Suffix column missing: 0
  Both values present: 547,162
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair: LivingArea vs LivingArea.1
  Base column missing: 552
  Suffix column missing: 552
  Both values present: 546,610
  Both present but different: 0
  Equal including matching missing values: 547,162 / 547,162

Pair

## Step 4: Investigate Column Differences

Most `.1` columns in the Listings dataset are exact duplicates. Two exceptions need special handling:

- `BuyerOfficeName` has one conflicting non-missing value.
- `UnparsedAddress` has one row where the original and suffixed columns differ in missingness.

This step identifies those records using `ListingKey` where available. We will use the results to decide how to preserve information before dropping redundant columns.

In [6]:
# Inspect the one BuyerOfficeName conflict.
print("=== BUYER OFFICE NAME CONFLICT ===")

buyer_conflict = (
    listings["BuyerOfficeName"].notna()
    & listings["BuyerOfficeName.1"].notna()
    & listings["BuyerOfficeName"].ne(listings["BuyerOfficeName.1"])
)

buyer_cols = [
    col for col in [
        "ListingKey",
        "PropertyType",
        "CloseDate",
        "BuyerOfficeName",
        "BuyerOfficeName.1",
    ]
    if col in listings.columns
]

print(f"Conflicting rows: {int(buyer_conflict.sum()):,}")
print(listings.loc[buyer_conflict, buyer_cols].to_string(index=False))


# Inspect rows where UnparsedAddress and UnparsedAddress.1
# have different missing-value patterns.
print("\n=== UNPARSED ADDRESS MISSINGNESS DIFFERENCE ===")

address_difference = (
    listings["UnparsedAddress"].isna()
    != listings["UnparsedAddress.1"].isna()
)

address_cols = [
    col for col in [
        "ListingKey",
        "PropertyType",
        "UnparsedAddress",
        "UnparsedAddress.1",
        "Latitude",
        "Longitude",
    ]
    if col in listings.columns
]

print(f"Rows with different missingness: {int(address_difference.sum()):,}")
print(
    listings.loc[address_difference, address_cols]
    .to_string(index=False)
)

=== BUYER OFFICE NAME CONFLICT ===
Conflicting rows: 1
 ListingKey PropertyType CloseDate BuyerOfficeName BuyerOfficeName.1
 1151935665  Residential       NaN         Compass                Co

=== UNPARSED ADDRESS MISSINGNESS DIFFERENCE ===
Rows with different missingness: 1
 ListingKey PropertyType UnparsedAddress UnparsedAddress.1  Latitude   Longitude
 1151935665  Residential 1369 Milton Ave               NaN  37.88143 -122.044742


## Step 5: Remove Redundant Columns Without Losing Information

The audit confirmed that most suffixed columns duplicate their original columns exactly. We will:

1. Create independent working copies of Sold and Listings.
2. Remove the nine verified duplicate columns.
3. Keep `UnparsedAddress` as the canonical address column, checking the suffix for any values that might need to be recovered.
4. Rename `BuyerOfficeName.1` to `BuyerOfficeName_suffix_review` so the conflicting value remains available for investigation.
5. Confirm that row counts and the mortgage-rate field are unchanged.

No records will be deleted in this step.

In [7]:
# Create working copies. The original DataFrames remain unchanged.
sold_clean = sold.copy()
listings_clean = listings.copy()

# These nine columns were confirmed to match their base columns
# across all rows, including missing values.
verified_duplicate_columns = [
    "PropertyType.1",
    "ListAgentFirstName.1",
    "DaysOnMarket.1",
    "LivingArea.1",
    "Longitude.1",
    "Latitude.1",
    "ListPrice.1",
    "ListAgentLastName.1",
    "CloseDate.1",
]

# Remove only the verified duplicates that exist.
columns_to_drop = [
    col for col in verified_duplicate_columns
    if col in listings_clean.columns
]
listings_clean = listings_clean.drop(columns=columns_to_drop)

# Keep the canonical address. Recover a value from the suffix only
# if the canonical address is missing and the suffix has a value.
if "UnparsedAddress.1" in listings_clean.columns:
    address_suffix = listings_clean["UnparsedAddress.1"]

    recover_address = (
        listings_clean["UnparsedAddress"].isna()
        & address_suffix.notna()
    )

    recovered_count = int(recover_address.sum())

    listings_clean.loc[
        recover_address, "UnparsedAddress"
    ] = address_suffix.loc[recover_address]

    listings_clean = listings_clean.drop(
        columns=["UnparsedAddress.1"]
    )
else:
    recovered_count = 0

# Preserve the conflicting BuyerOfficeName value for review.
if "BuyerOfficeName.1" in listings_clean.columns:
    listings_clean = listings_clean.rename(
        columns={
            "BuyerOfficeName.1": "BuyerOfficeName_suffix_review"
        }
    )

# Verify row counts, columns, and mortgage-rate retention.
print("=== COLUMN CLEANUP VALIDATION ===")

for name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    original = datasets[name]

    print(f"\n{name}")
    print(f"Original rows: {len(original):,}")
    print(f"Working-copy rows: {len(df):,}")
    print(f"Original columns: {len(original.columns):,}")
    print(f"Working-copy columns: {len(df.columns):,}")
    print(f"Row count unchanged: {len(original) == len(df)}")
    print(f"Mortgage-rate field retained: {'rate_30yr_fixed' in df.columns}")

print(f"\nRedundant columns removed: {len(columns_to_drop) + 1}")
print(f"Address values recovered from suffix: {recovered_count}")
print(
    "Buyer office review column retained:",
    "BuyerOfficeName_suffix_review" in listings_clean.columns
)
print(
    "Buyer office conflict record:",
    listings_clean.loc[
        listings_clean["ListingKey"].astype(str) == "1151935665",
        [
            "ListingKey",
            "BuyerOfficeName",
            "BuyerOfficeName_suffix_review",
            "UnparsedAddress",
        ],
    ].to_string(index=False)
)

=== COLUMN CLEANUP VALIDATION ===

Sold
Original rows: 414,003
Working-copy rows: 414,003
Original columns: 86
Working-copy columns: 86
Row count unchanged: True
Mortgage-rate field retained: True

Listings
Original rows: 547,162
Working-copy rows: 547,162
Original columns: 86
Working-copy columns: 76
Row count unchanged: True
Mortgage-rate field retained: True

Redundant columns removed: 10
Address values recovered from suffix: 0
Buyer office review column retained: True
Buyer office conflict record:  ListingKey BuyerOfficeName BuyerOfficeName_suffix_review UnparsedAddress
 1151935665         Compass                            Co 1369 Milton Ave


## Step 6: Standardize Data Types

Consistent data types are essential for accurate calculations, date comparisons, and visualizations.

In this step, I will:

- Convert `CloseDate`, `PurchaseContractDate`, `ListingContractDate`, and `ContractStatusChangeDate` to pandas datetime types.
- Convert key price, property-size, bedroom, bathroom, days-on-market, coordinate, and mortgage-rate fields to numeric types.
- Record original missing values separately from non-missing values that fail conversion.
- Preserve all rows and retain the original input DataFrames.

I will not fill missing values or remove records in this step. Any conversion failures will be documented for review.

In [8]:
# Fields that should represent dates.
date_fields = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]

# Fields used in analysis and quality checks.
numeric_fields = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "Latitude",
    "Longitude",
    "rate_30yr_fixed",
]

conversion_log = []

for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(f"\n{'=' * 70}")
    print(f"{dataset_name.upper()} — TYPE STANDARDIZATION")
    print("=" * 70)

    # Convert date fields.
    for col in date_fields:
        if col not in df.columns:
            continue

        original = df[col]
        original_nonmissing = original.notna()

        converted = pd.to_datetime(original, errors="coerce")

        missing_before = int(original.isna().sum())
        conversion_failures = int(
            (original_nonmissing & converted.isna()).sum()
        )

        df[col] = converted

        conversion_log.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "datetime",
            "missing_before": missing_before,
            "conversion_failures": conversion_failures,
            "missing_after": int(df[col].isna().sum()),
            "final_dtype": str(df[col].dtype),
        })

    # Convert numeric fields.
    for col in numeric_fields:
        if col not in df.columns:
            continue

        original = df[col]
        original_nonmissing = original.notna()

        converted = pd.to_numeric(original, errors="coerce")

        missing_before = int(original.isna().sum())
        conversion_failures = int(
            (original_nonmissing & converted.isna()).sum()
        )

        df[col] = converted

        conversion_log.append({
            "dataset": dataset_name,
            "column": col,
            "target_type": "numeric",
            "missing_before": missing_before,
            "conversion_failures": conversion_failures,
            "missing_after": int(df[col].isna().sum()),
            "final_dtype": str(df[col].dtype),
        })

# Display a detailed conversion summary.
conversion_summary = pd.DataFrame(conversion_log)

print("\n=== CONVERSION SUMMARY ===")
print(conversion_summary.to_string(index=False))

print("\n=== FINAL DATE AND NUMERIC DTYPES ===")
for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(f"\n{dataset_name}")
    for col in date_fields + numeric_fields:
        if col in df.columns:
            print(f"{col}: {df[col].dtype}")

print("\n=== VALIDATION ===")
for dataset_name, df in [
    ("Sold", sold_clean),
    ("Listings", listings_clean),
]:
    print(
        f"{dataset_name}: {len(df):,} rows; "
        f"rate_30yr_fixed present = {'rate_30yr_fixed' in df.columns}"
    )

print(
    "\nTotal conversion failures:",
    f"{conversion_summary['conversion_failures'].sum():,}"
)


SOLD — TYPE STANDARDIZATION

LISTINGS — TYPE STANDARDIZATION

=== CONVERSION SUMMARY ===
 dataset                   column target_type  missing_before  conversion_failures  missing_after    final_dtype
    Sold                CloseDate    datetime               0                    0              0 datetime64[ns]
    Sold     PurchaseContractDate    datetime             194                    0            194 datetime64[ns]
    Sold      ListingContractDate    datetime               1                    0              1 datetime64[ns]
    Sold ContractStatusChangeDate    datetime             594                    0            594 datetime64[ns]
    Sold               ClosePrice     numeric               0                    0              0        float64
    Sold                ListPrice     numeric               0                    0              0        float64
    Sold               LivingArea     numeric             234                    0            234        float64
    So

## Step 7: Identify Invalid Numeric Values

This step checks the key numeric fields against basic validity rules:

- `ClosePrice` must be greater than zero when present.
- `LivingArea` must be greater than zero when present.
- `DaysOnMarket` must not be negative.
- `BedroomsTotal` and `BathroomsTotalInteger` must not be negative.

For each rule, we will add a Boolean flag column. Missing values will not be classified as invalid by these flags; missingness will be summarized separately.

We will retain all records, including those with flagged values, to preserve data for review and document the quality issues transparently.